# Lesson 14 — Exercises

Run `setup.sql` first, then work through `lesson.ipynb`. Solutions are in `exercises-solutions.ipynb`.

Work the exercises in order. Exercise 2 uses `DBCC PAGE`, which needs a real FileId/PageId — read it from your Exercise 1 output and substitute it into the commented `DBCC PAGE` line.

In [ ]:
USE AdventureWorks2022;
GO

### Exercise 1

For SalesOrderID 43659 in `lesson14.SalesOrderCI`, report its physical location (file:page:slot). Then list all DATA_PAGE pages of the table.

**Expected:** one location string, then a list of data page ids.

**Hint:** `%%physloc%%` + `sys.fn_PhysLocFormatter`; `sys.dm_db_database_page_allocations`.

In [ ]:
SELECT SalesOrderID, sys.fn_PhysLocFormatter(%%physloc%%) AS PhysicalLocation
FROM lesson14.SalesOrderCI
WHERE SalesOrderID = 43659;

SELECT allocated_page_file_id AS FileId, allocated_page_page_id AS PageId, page_type_desc
FROM sys.dm_db_database_page_allocations(
        DB_ID(), OBJECT_ID('lesson14.SalesOrderCI'), NULL, NULL, 'DETAILED')
WHERE page_type_desc = 'DATA_PAGE'
ORDER BY PageId;

**Why:** `%%physloc%%` gives the row's RID; page allocations DMV lists every page.

### Exercise 2

Using `DBCC PAGE`, crack open the data page that holds SalesOrderID 43659 (use the PageId you found in Exercise 1). Turn on trace flag 3604 first.

In [ ]:
-- Replace 1 and 0 below with the FileId/PageId from Exercise 1's PhysicalLocation.
DBCC TRACEON(3604);
-- DBCC PAGE ('AdventureWorks2022', 1, 0, 3);   -- <-- substitute real FileId/PageId
DBCC TRACEOFF(3604);

**Why:** read file/page from the physloc formatter output (file:page:slot), then DBCC PAGE. The output shows the page header (m_type=1 data page) and each slot's row contents.

### Exercise 3

Measure fragmentation of `lesson14.SalesOrderCI`'s clustered index. Then REBUILD it and measure again. Report `avg_fragmentation_in_percent` before and after.

In [ ]:
SELECT 'before' AS Phase, avg_fragmentation_in_percent, page_count
FROM sys.dm_db_index_physical_stats(
        DB_ID(), OBJECT_ID('lesson14.SalesOrderCI'), 1, NULL, 'DETAILED');

ALTER INDEX PK_lesson14_SalesOrderCI ON lesson14.SalesOrderCI REBUILD;

SELECT 'after' AS Phase, avg_fragmentation_in_percent, page_count
FROM sys.dm_db_index_physical_stats(
        DB_ID(), OBJECT_ID('lesson14.SalesOrderCI'), 1, NULL, 'DETAILED');

**Why:** REBUILD recreates the index contiguously, dropping fragmentation toward 0%.

### Exercise 4

`lesson14.SalesOrderHeap` is a heap. Report its `forwarded_record_count`. Explain in a comment why forwarded records hurt read performance.

In [ ]:
SELECT forwarded_record_count, page_count
FROM sys.dm_db_index_physical_stats(
        DB_ID(), OBJECT_ID('lesson14.SalesOrderHeap'), 0, NULL, 'DETAILED');

**Why:** index_id 0 = the heap. Forwarded records appear after in-place updates grow rows. A forwarded record leaves a pointer at the original slot to the new location. A scan must follow the pointer (an extra random I/O) to read the row, so reads do more work. Fix: give the table a clustered index (B-trees relocate via the key, not forwarding pointers), or run ALTER TABLE ... REBUILD to remove existing forwarding pointers.

### Exercise 5

Add 3 more middle-valued keys to `lesson14.SplitDemo` (e.g. 5, 12, 18) and show that page_count increased compared to before the insert. In a comment, explain what a page split is and one way to reduce them.

In [ ]:
SELECT 'before' AS Phase, page_count FROM sys.dm_db_index_physical_stats(
        DB_ID(), OBJECT_ID('lesson14.SplitDemo'), 1, NULL, 'DETAILED');

INSERT lesson14.SplitDemo (Id) VALUES (5), (12), (18);

SELECT 'after' AS Phase, page_count FROM sys.dm_db_index_physical_stats(
        DB_ID(), OBJECT_ID('lesson14.SplitDemo'), 1, NULL, 'DETAILED');

**Why:** middle-valued inserts split full pages; page_count rises. A page split happens when a row must go onto a full page: SQL Server allocates a new page, moves ~half the rows to it, then inserts. It causes fragmentation and extra log. Reduce splits with a lower FILLFACTOR (leave free space) or an ever-increasing key (appends go to the end and rarely split mid-index).